In [1]:
from itertools import product
class Factor:
    def __init__(self, vars, table):
        self.vars, self.table = vars, table

def make_factor(V, e, bn):
    node = bn[V]
    vars = [p for p in node["parents"] if p not in e]
    if V not in e:
        vars.append(V)
    table = {}
    for values in product([True, False], repeat=len(vars)):
        a = dict(zip(vars, values))
        a.update(e)
        p = node["cpt"][tuple(a[x] for x in node["parents"])]
        table[values] = p if a[V] else 1 - p
    return Factor(vars, table)

def multiply(f1, f2):
    vars = list(dict.fromkeys(f1.vars + f2.vars))
    table = {}
    for values in product([True, False], repeat=len(vars)):
        a = dict(zip(vars, values))
        k1 = tuple(a[x] for x in f1.vars)
        k2 = tuple(a[x] for x in f2.vars)
        table[values] = f1.table[k1] * f2.table[k2]
    return Factor(vars, table)

def sum_out(V, factors):
    fs = [f for f in factors if V in f.vars]
    others = [f for f in factors if V not in f.vars]
    if not fs:
        return others
    f = fs[0]
    for g in fs[1:]:
        f = multiply(f, g)
    vars = [x for x in f.vars if x != V]
    table = {}
    for values in product([True, False], repeat=len(vars)):
        a = dict(zip(vars, values))
        table[values] = sum(
            f.table[tuple((v if x == V else a[x]) for x in f.vars)] for v in [True, False]
        )
    return others + [Factor(vars, table)]

def elimination_ask(X, e, bn, order):
    factors = [make_factor(V, e, bn) for V in bn]
   
    for V in order:
        if V != X and V not in e:
            factors = sum_out(V, factors)
           
    f = factors[0]
    for g in factors[1:]:
        f = multiply(f, g)
       
    result = {}
    for x in [True, False]:
        k = tuple(x if v == X else e.get(v) for v in f.vars)
        result[x] = f.table[k]
       
    s = sum(result.values())
    return {k: v / s for k, v in result.items()}

bn = {
    "R": {"parents": [], "cpt": {(): 0.30}},
    "A": {"parents": [], "cpt": {(): 0.005}},
    "T": {"parents": ["R", "A"], "cpt": {
        (True, True): 0.90, (True, False): 0.70, (False, True): 0.60, (False, False): 0.10}},
    "D": {"parents": ["T"], "cpt": {
        (True,): 0.80, (False,): 0.20}},
    "M": {"parents": ["T"], "cpt": {
        (True,): 0.50, (False,): 0.05}}
}

result = elimination_ask(
    "A", {"D": False, "M": True}, bn, ["M","D","T","A","R"]
)

print("P(A | D=False, M=True)")
print("A=True :", round(result[True], 4))
print("A=False:", round(result[False], 4))

P(A | D=False, M=True)
A=True : 0.0072
A=False: 0.9928


In [3]:
P_B = 0.001
P_E = 0.002

P_A = {
    (True, True): 0.95,
    (True, False): 0.94,
    (False, True): 0.29,
    (False, False): 0.001
}

P_J = {True: 0.90, False: 0.05}
P_M = {True: 0.70, False: 0.01}

T, F = True, False

# Step 1: Factors for evidence (J = True, M = True)
f_J = {a: P_J[a] for a in (T, F)}
f_M = {a: P_M[a] for a in (T, F)}

# Step 2: Multiply factors
f_JM = {
    a: f_J[a] * f_M[a]
    for a in (T, F)
}

# Step 3: Sum out A
f_A = {
    (b, e): sum(
        (P_A[(b, e)] if a else 1 - P_A[(b, e)]) * f_JM[a]
        for a in (T, F)
    )
    for b in (T, F)
    for e in (T, F)
}

# Step 4: Sum out E
f_E = {
    b: sum(
        (P_E if e else 1 - P_E) * f_A[(b, e)]
        for e in (T, F)
    )
    for b in (T, F)
}

# Step 5: Multiply by P(B) and normalize
unnorm = {
    b: (P_B if b else 1 - P_B) * f_E[b]
    for b in (T, F)
}

Z = unnorm[T] + unnorm[F]

# Final Answer
print("P(Burglary | John, Mary)     =", unnorm[T] / Z)
print("P(Not Burglary | John, Mary) =", unnorm[F] / Z)

P(Burglary | John, Mary)     = 0.2841718353643929
P(Not Burglary | John, Mary) = 0.7158281646356071
